# Train the XGBoost triage model on HI-Small

This notebook builds account-day features from `HI-Small_Trans.csv`, splits them by time, trains XGBoost with early stopping, and evaluates once on the test block.

- **Split:** the first 75% of usable days is the development block. Its last 20% is validation, and the first 80% of it is train. The last 25% is test. Each cut has a purge gap of `HORIZON` days, so no label window crosses a split.
- **Early stopping:** each grid point stops when validation PR-AUC has not improved for 50 rounds. The best grid point is kept.
- **Scope:** HI-Small is a pipeline check. Its activity falls mostly on days 1 to 10, so the test block is about four days. Treat the metrics as a smoke test. The real evaluation runs on HI-Large.
- **Placeholders:** `BUDGET` (flagged account-days per day) stays at 50 until the alert budget is confirmed.

The feature and training code lives in `triage_lib/`, copied from `backend/app/ml/`. This notebook is the training source of truth.

## 1. Install pinned packages

The versions match the ones verified in the backend venv. If the install changes numpy or pandas on a running kernel, restart the kernel and run from section 2.

In [ ]:
%pip install -q "pandas==3.0.6" "numpy==2.5.4" "xgboost==3.4.1" "pyarrow==26.0.0" "scikit-learn==1.9.1" "boto3"

## 2. Configuration

Run this notebook from the `notebooks/` folder. Paths are relative to it, so the same notebook works on SageMaker and locally.

In [ ]:
from pathlib import Path
import json
import shutil
import sys
import time

import numpy as np
import pandas as pd

NB_DIR = Path.cwd()  # run from the notebooks/ folder
sys.path.insert(0, str(NB_DIR))

# Data in S3 (uploaded once with aws s3 cp)
S3_BUCKET = "neu-datasets"
S3_KEY_HI_SMALL = "HI-Small_Trans.csv"
S3_KEY_PATTERNS = "HI-Small_Patterns.txt"  # optional; per-typology recall is skipped if it is missing

DATA_DIR = NB_DIR / "data"
FEATURES_DIR = NB_DIR / "features"
MODELS_DIR = NB_DIR / "models"
EVAL_DIR = NB_DIR / "eval"
LOCAL_CSV = DATA_DIR / S3_KEY_HI_SMALL
PATTERNS = DATA_DIR / S3_KEY_PATTERNS
for folder in (DATA_DIR, FEATURES_DIR, MODELS_DIR, EVAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RUN_NAME = "triage_hi_small"
HORIZON = 2           # label horizon in days; 7 would leave no test days on HI-Small
DEV_FRACTION = 0.75   # first 75% of usable days = development block (train + validation)
VAL_FRACTION = 0.20   # last 20% of the development block = validation
BUDGET = 50           # placeholder: flagged account-days per day for the alert gate
UPLOAD_RESULTS = True # copy model and metrics to s3://<bucket>/triage/<RUN_NAME>/ at the end

## 3. Download the data from S3

The CSV is downloaded once. If it is already in `data/`, the download is skipped.

In [ ]:
import boto3

s3 = boto3.client("s3")
if not LOCAL_CSV.exists():
    s3.download_file(S3_BUCKET, S3_KEY_HI_SMALL, str(LOCAL_CSV))
with LOCAL_CSV.open() as fh:
    n_rows = sum(1 for _ in fh) - 1
print(f"{LOCAL_CSV.name}: {LOCAL_CSV.stat().st_size / 2**20:.1f} MiB, {n_rows:,} transactions")

if not PATTERNS.exists():
    try:
        s3.download_file(S3_BUCKET, S3_KEY_PATTERNS, str(PATTERNS))
    except Exception as exc:  # the typology file is optional
        print(f"typology file not downloaded ({exc.__class__.__name__}); per-typology recall will be skipped")
patterns = PATTERNS if PATTERNS.exists() else None
print("typology file:", patterns or "none")

## 4. Build features

The rule pass runs once on the whole file. Each account bucket is then saved as it finishes, so an interrupted run resumes instead of starting over. The cache is cleared automatically if the CSV, horizon, or bucket count changes.

In [ ]:
from triage_lib.triage_features import FEATURES_FILE, MANIFEST_FILE, run_features

t0 = time.time()
manifest = run_features(LOCAL_CSV, FEATURES_DIR, horizon=HORIZON)
print(
    f"rows={manifest['rows']:,}  positives={manifest['positives']:,}  "
    f"days={manifest['max_day']}  horizon={manifest['horizon']}  buckets={manifest['buckets']}  "
    f"({time.time() - t0:.0f}s)"
)
print("features:", len(manifest["features"]), "columns:", manifest["features"])

## 5. Time split (75/25, with validation and purge gaps)

Days are numbered from the first calendar day in the file. A row for day `t` covers the 7 days ending at `t`. Each boundary is followed by a gap of `HORIZON` days, so no label crosses a split.

In [ ]:
usable = pd.read_parquet(FEATURES_DIR / FEATURES_FILE, columns=["t"])["t"]
max_t = int(usable.max())
dev_end = int(max_t * DEV_FRACTION)
train_end = int(dev_end * (1 - VAL_FRACTION))  # first 80% of the development block
val_end = dev_end                               # last 20% of the development block
assert 1 <= train_end and val_end - (train_end + HORIZON) >= 1 and max_t - (val_end + HORIZON) >= 1, \
    "not enough days for train, validation and test with this horizon"

print(f"usable days: 1..{max_t}")
print(f"train:      days 1..{train_end}")
print(f"gap:        {HORIZON} days")
print(f"validation: days {train_end + HORIZON + 1}..{val_end}")
print(f"gap:        {HORIZON} days")
print(f"test:       days {val_end + HORIZON + 1}..{max_t}")

## 6. Train with early stopping

Training uses 10:1 negatives, downsampled in the train block only. The grid is 12 points: `max_depth` 4/6/8, `eta` 0.05/0.1, `min_child_weight` 1/5. Each point stops early on validation PR-AUC. The test block is not used here.

In [ ]:
from triage_lib.triage_train import run_train

t0 = time.time()
metrics = run_train(
    FEATURES_DIR,
    EVAL_DIR / RUN_NAME,
    budget=BUDGET,
    train_end=train_end,
    val_end=val_end,
    patterns_path=patterns,
)
print(
    f"best params: {metrics['best_params']}  best_iteration={metrics['best_iteration']}  "
    f"val PR-AUC={metrics['val_aucpr']:.4f}  ({time.time() - t0:.0f}s)"
)
print("rows:", metrics["rows"])
print("positives:", metrics["positives"])
pd.DataFrame(metrics["grid"]).sort_values("val_aucpr", ascending=False)

## 7. Evaluate once on the test block

At a fixed budget of flagged account-days per day, the model and the app's rule score are compared on the same rows.

In [ ]:
summary = pd.DataFrame([
    {"scorer": "XGBoost", **metrics["test"]["model"]},
    {"scorer": "app rule score", **metrics["test"]["rules_app"]},
]).set_index("scorer")
display(summary[["budget_per_day", "days", "flagged", "tp", "fp", "positives",
                 "precision_at_budget", "recall_at_budget", "pr_auc"]])
print("test rows:", metrics["rows"]["test"], "  positives:", metrics["positives"]["test"])
print("suggested score threshold from validation:", metrics["suggested_threshold_val"])
if metrics.get("typology_recall_test"):
    display(pd.DataFrame(metrics["typology_recall_test"]).T)
else:
    print("per-typology recall skipped (no HI-Small_Patterns.txt)")

## 8. Save the model and metrics

The model goes to `models/<RUN_NAME>/model.json`. The metrics stay in `eval/<RUN_NAME>/metrics.json`. If `UPLOAD_RESULTS` is on, both are also copied to S3. The execution role needs `s3:PutObject` on the bucket for that step.

In [ ]:
model_dir = MODELS_DIR / RUN_NAME
model_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(EVAL_DIR / RUN_NAME / "model.json", model_dir / "model.json")
print("model:  ", model_dir / "model.json")
print("metrics:", EVAL_DIR / RUN_NAME / "metrics.json")

if UPLOAD_RESULTS:
    for local, key in [
        (model_dir / "model.json", f"triage/{RUN_NAME}/model.json"),
        (EVAL_DIR / RUN_NAME / "metrics.json", f"triage/{RUN_NAME}/metrics.json"),
    ]:
        s3.upload_file(str(local), S3_BUCKET, key)
        print(f"uploaded s3://{S3_BUCKET}/{key}")

## 9. Feature importance (gain)

In [ ]:
importance = pd.Series(metrics["top_features_gain"]).sort_values(ascending=False)
importance.head(20).to_frame("gain")

## 10. Reason codes for three flagged test rows

The reason codes are the features that pushed each score up most, from XGBoost's own contributions. They are the evidence the investigator and the report agent see. The LLM may cite them but must not invent patterns.

In [ ]:
import xgboost as xgb

from triage_lib.explain import reason_codes
from triage_lib.triage_train import assign_splits, predict

features = json.loads((FEATURES_DIR / MANIFEST_FILE).read_text(encoding="utf-8"))["features"]
frame = pd.read_parquet(FEATURES_DIR / FEATURES_FILE)
frame["split"] = assign_splits(frame["t"].to_numpy(), HORIZON, train_end, val_end)
test = frame[frame["split"] == "test"].copy()

booster = xgb.Booster()
booster.load_model(str(EVAL_DIR / RUN_NAME / "model.json"))
test["score"] = predict(booster, test, features)
examples = test.sort_values("score", ascending=False).head(3)
for (_, row), codes in zip(examples.iterrows(), reason_codes(booster, examples, features)):
    print(f"{row['acct']}  {row['t_date'].date()}  score={row['score']:.3f}  top features: {codes}")

## Next steps

1. Treat these numbers as a pipeline check only. The test block on HI-Small is about four days.
2. Run the same notebook on HI-Large for the real model. Change `S3_KEY_HI_SMALL` and `RUN_NAME`, and use a larger instance (see `README.md`).
3. The outside test on SAML-D needs a schema adapter first. It is not in this notebook.
4. Set `BUDGET` once the alert budget (cases per day) is confirmed.